In [ ]:
import requests
import os
from dotenv import load_dotenv
from modules.get_token import get_token
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
def inspect_graphql_schema(token):
    """Introspect the GraphQL schema to find available fields"""
    url = f"{start_url}/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    # Introspection query to get schema information
    introspection_query = """
    {
        __type(name: "CustomerPO") {
            name
            fields(includeDeprecated: false) {
                name
                type {
                    name
                    kind
                    ofType {
                        name
                        kind
                        ofType {
                            name
                            kind
                        }
                    }
                }
            }
        }
    }
    """
    
    try:
        payload = {"query": introspection_query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        res.raise_for_status()
        
        data = res.json()
        
        if 'errors' in data:
            print("GraphQL Errors:", data['errors'])
            return
        
        if data['data']['__type'] is None:
            print("Type 'CustomerPO' not found. Let me search for available types...")
            return search_types(token)
        
        print("\n" + "="*60)
        print("AVAILABLE FIELDS IN CustomerPO:")
        print("="*60)
        
        for field in data['data']['__type']['fields']:
            field_name = field['name']
            field_type = get_type_string(field['type'])
            print(f"  • {field_name}: {field_type}")
        
        print("\n" + "="*60)
        print("NESTED OBJECTS (may need sub-selections):")
        print("="*60)
        
        for field in data['data']['__type']['fields']:
            field_type = field['type']
            if field_type['kind'] == 'OBJECT' or (field_type.get('ofType', {}).get('kind') == 'OBJECT'):
                print(f"\n  → {field['name']} is an OBJECT - you need to specify which fields to query from it")
                # Try to get fields from the nested object
                inspect_nested_type(token, field['name'], field_type)
        
    except Exception as e:
        print(f"Error: {e}")

def get_type_string(type_obj):
    """Convert GraphQL type to readable string"""
    if type_obj['kind'] == 'NON_NULL':
        return get_type_string(type_obj['ofType']) + "!"
    elif type_obj['kind'] == 'LIST':
        return "[" + get_type_string(type_obj['ofType']) + "]"
    else:
        return type_obj['name'] or 'Unknown'

def inspect_nested_type(token, field_name, field_type):
    """Introspect a nested object type"""
    # Get the actual type name
    type_name = field_type['name']
    if not type_name and field_type.get('ofType'):
        type_name = field_type['ofType']['name']
    
    if not type_name:
        return
    
    url = f"https://msp.adionsystems.com/api/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    query = f"""
    {{
        __type(name: "{type_name}") {{
            name
            fields(includeDeprecated: false) {{
                name
                type {{
                    name
                    kind
                    ofType {{
                        name
                        kind
                    }}
                }}
            }}
        }}
    }}
    """
    
    try:
        payload = {"query": query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        data = res.json()
        
        if data.get('data', {}).get('__type'):
            print(f"\n    Fields available in {type_name}:")
            for field in data['data']['__type']['fields']:
                print(f"      - {field['name']}: {get_type_string(field['type'])}")
    except Exception as e:
        print(f"    Could not inspect {type_name}: {e}")

def search_types(token):
    """Search for available types in the schema"""
    url = f"https://msp.adionsystems.com/api/graphql?token={token}"
    header = {"content-type": "application/json"}
    
    query = """
    {
        __schema {
            types {
                name
                kind
            }
        }
    }
    """
    
    try:
        payload = {"query": query}
        res = requests.post(url, headers=header, json=payload, timeout=10)
        data = res.json()
        
        print("\nAvailable types in schema:")
        for type_obj in data['data']['__schema']['types']:
            if not type_obj['name'].startswith('__'):
                print(f"  • {type_obj['name']} ({type_obj['kind']})")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    token = get_token()
    if token:
        inspect_graphql_schema(token)


AVAILABLE FIELDS IN CustomerPO:
  • bulkImportId: String
  • buyer: String
  • client: Contact
  • clientPlainText: String
  • clientPONumber: String
  • clientsCountry: String
  • confirmationNotes: String
  • confirmationSent: Boolean
  • confirmationSentBy: User
  • confirmationSentByPlainText: String
  • confirmationSentDate: String
  • copiedFrom: String
  • copiedTime: String
  • createdBy: User
  • createdByPlainText: String
  • createdTime: String
  • currency: String
  • currencyLocale: String
  • dateEntered: String
  • editLog: EditLog
  • editLogPlainText: String
  • exchangeRate: String
  • fob: String
  • installGroup: String
  • isComplete: String
  • isTemplate: Boolean
  • lastModifiedBy: User
  • lastModifiedByApplication: Authorization
  • lastModifiedByPlainText: String
  • lastModifiedTime: String
  • legacyId: String!
  • notes: String
  • originalSortPosition: Int
  • partsOrdered: PaginatedCustomerPOPartOrderedResult!
  • paymentTerms: String
  • paymentTermsDi

In [5]:
def fetch_complete_order_flow(token, po_number):
    """Fetch complete flow: CustomerPO → WorkOrder → Invoice"""
    
    query = """
    query completeOrderFlow($poNumber: String) {
        # 1. Get Customer PO
        customerPOs(filter: {clientPONumber: $poNumber}) {
            records {
                id
                clientPONumber
                client { id name }
                totalAmount
                dueDate
                createdTime
                partsOrdered {
                    part { id number }
                    quantity
                }
            }
        }
        
        # 2. Get Work Orders linked to this PO
        workOrders(filter: {customerPONumber: $poNumber}) {
            records {
                workOrderNumber
                customerPONumber
                customer { name }
                quantityOrdered
                dueDate
                status
                part { id number }
                
                # Manufacturing costs
                costDisbursement {
                    records {
                        totalCost
                    }
                }
            }
        }
        
        # 3. Get Invoices linked to this PO
        invoices(filter: {clientPoNum: $poNumber}) {
            records {
                invoiceId
                clientPoNum
                invoiceDate
                invoicedDollars
                status
                items {
                    records {
                        part { id number }
                        quantity
                        unitPrice
                    }
                }
            }
        }
    }
    """
    
    # Execute and combine data
    # This gives you complete traceability:
    # Customer PO → Work Orders → Invoices